# TransformerLens Main Demo Notebook

> **Source:** [TransformerLens Official Docs](https://transformerlensorg.github.io/TransformerLens/generated/demos/Main_Demo.html)  
> **Author:** Neel Nanda  
> **Library:** [TransformerLens](https://github.com/TransformerLensOrg/TransformerLens)

---

**To use this notebook with a GPU (VS Code):** connect to your GPU VM with *Remote-SSH*, open this notebook there, and pick the Python environment that has PyTorch with CUDA as the kernel (top-right **Select Kernel**). Expect roughly 1.5-2 GB of VRAM in total.

This is a reference notebook covering the main features of TransformerLens for **mechanistic interpretability**. The goal of mechanistic interpretability is to take a trained model and reverse engineer the algorithms the model learned during training from its weights.

**Tips:**
- Run all cells from top to bottom
- Graphs are interactive (Plotly)
- See [ARENA chapter](https://learn.arena.education/chapter1_transformer_interp/) for a gentler structured introduction

---

## Table of Contents
1. [Setup](#1-setup)
2. [Introduction & Loading Models](#2-introduction--loading-models)
3. [Caching All Activations](#3-caching-all-activations)
4. [Hooks: Intervening on Activations](#4-hooks-intervening-on-activations)
   - 4a. Head Ablation
   - 4b. Activation Patching on IOI Task
5. [Hooks: Accessing Activations (Induction Heads)](#5-hooks-accessing-activations--induction-heads)
6. [Available Models](#6-available-models)
7. [Model Architecture Conventions & Parameter Names](#7-model-architecture-conventions--parameter-names)

---
## 1. Setup

Install dependencies (only runs if you're in Google Colab), then import everything we need.

In [2]:
import os
import sys

# Set to True once on a fresh VM / virtual environment. Installs into *this kernel's* environment.
INSTALL_DEPS = False
if INSTALL_DEPS:
    %pip install -q transformer_lens circuitsvis

# Only used further down to make a demo deterministic in CI.
IN_GITHUB = os.getenv("GITHUB_ACTIONS") == "true"

In [3]:
!pip install plotly
# Plotly renderer for VS Code notebooks (also works over Remote-SSH).
# If charts come out blank, try "notebook_connected" instead.
import plotly.io as pio

pio.renderers.default = "vscode"

print(f"Using renderer: {pio.renderers.default}")

Using renderer: vscode


In [4]:
!pip install circuitsvis
import circuitsvis as cv

# Quick sanity check that CircuitsVis is working
cv.examples.hello("Neel")

In [5]:
!pip install einops
!pip install fancy_einsum
!pip install tqdm
!pip install jaxtyping
# Core imports
import torch
import torch.nn as nn
import einops
from fancy_einsum import einsum
import tqdm.auto as tqdm
import plotly.express as px
from jaxtyping import Float
from functools import partial

/home/ubuntu/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
!pip install transformer_lens
# TransformerLens imports
import transformer_lens.utilities as utils
from transformer_lens.hook_points import HookPoint   # Hook point utilities
from transformer_lens import FactoredMatrix
from transformer_lens.model_bridge import TransformerBridge

In [7]:
# Turn off autograd — we're doing inference only, not training.
# This saves GPU memory significantly.
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [8]:
# -----------------------------------------------------------------------
# Plotting helper functions
# These wrap Plotly Express with sensible defaults for interpretability work
# -----------------------------------------------------------------------

def imshow(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    """Display a 2D tensor as a heatmap (red = positive, blue = negative)."""
    px.imshow(
        utils.to_numpy(tensor),
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu",
        labels={"x": xaxis, "y": yaxis},
        **kwargs,
    ).show(renderer)

def line(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    """Display a 1D tensor as a line chart."""
    px.line(
        y=utils.to_numpy(tensor),
        labels={"x": xaxis, "y": yaxis},
        **kwargs,
    ).show(renderer)

def scatter(x, y, xaxis="", yaxis="", caxis="", renderer=None, **kwargs):
    """Display a scatter plot from two tensors."""
    x = utils.to_numpy(x)
    y = utils.to_numpy(y)
    px.scatter(
        y=y, x=x,
        labels={"x": xaxis, "y": yaxis, "color": caxis},
        **kwargs,
    ).show(renderer)

---
## 2. Introduction & Loading Models

TransformerLens supports **9,000+ models across 50+ architecture families**. You can load any of them with `TransformerBridge.boot_transformers(MODEL_NAME)`.

For this demo we use **GPT-2 Small** (80M parameters). The goal is to be able to dig into the internals of a model with surgical precision — inspect activations, intervene on them, and understand *why* the model produces certain outputs.

**Key idea:** TransformerLens wraps a HuggingFace model and instruments every internal activation with "hook points" — named locations where you can read or write any tensor during the forward pass.

### Supported Model Families (partial list)
| Family | Examples |
|--------|----------|
| GPT-2 | `openai-community/gpt2`, `gpt2-medium`, `gpt2-large`, `gpt2-xl` |
| Llama | `meta-llama/Llama-2-7b-hf`, Llama-3 variants |
| Gemma | `google/gemma-2b`, `google/gemma-2-9b` |
| Mistral | `mistralai/Mistral-7B-v0.1` |
| Toy models | `gelu-1l` through `gelu-4l`, `solu-1l` through `solu-12l` |
| + many more | Falcon, Qwen, DeepSeek, etc. |

In [9]:
# Detect GPU/CPU automatically
device = utils.get_device()
print(f"Using device: {device}")

Using device: cuda


In [10]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    free, total = torch.cuda.mem_get_info()
    print(f"GPU            : {props.name}")
    print(f"Total VRAM     : {total / 2**30:.1f} GiB")
    print(f"Free VRAM      : {free / 2**30:.1f} GiB")
    print(f"Torch / CUDA   : {torch.__version__} / {torch.version.cuda}")
else:
    print("No CUDA GPU visible to this kernel -- running on CPU.")

GPU            : NVIDIA RTX A4000
Total VRAM     : 14.6 GiB
Free VRAM      : 11.7 GiB
Torch / CUDA   : 2.11.0+cu128 / 12.8


In [11]:
# Load GPT-2 Small.
# boot_transformers() downloads the HuggingFace weights and wraps them in TransformerLens.
# enable_compatibility_mode() gives HookedTransformer-equivalent numerics for legacy code.
model = TransformerBridge.boot_transformers("openai-community/gpt2", device=device)
model.enable_compatibility_mode(disable_warnings=True)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5229.59it/s]


In [12]:
# You can run the model on a raw string.
# return_type options: "logits", "loss", "both", None
model_description_text = """## Loading and Running Models

TransformerLens supports 9,000+ models across 50+ architecture families. You can load any of them in with `TransformerBridge.boot_transformers(MODEL_NAME)`. For this demo notebook we'll look at GPT-2 Small, an 80M parameter model, see the Available Models section for info on the rest.

To try the model out, let's find the loss on this text!"""

loss = model(model_description_text, return_type="loss")
print("Model loss:", loss)
# Lower loss = model found this text more predictable

Model loss: tensor(4.5417, device='cuda:0')


---
## 3. Caching All Activations

**`model.run_with_cache(tokens)`** runs a forward pass AND stores *every* intermediate activation in a dictionary. This is the first thing you reach for when exploring a model's internals.

### What is the ActivationCache?
It's a dictionary-like object where keys are activation names like:
- `"resid_pre"` — residual stream *before* each layer
- `"resid_post"` — residual stream *after* each layer  
- `"attn.hook_pattern"` — attention weights (shape: [heads, dest_pos, src_pos])
- `"mlp_out"` — MLP output for each layer

You index it with `cache[activation_name, layer_index]`.

### Note on `remove_batch_dim`
Every activation has a leading batch dimension. With a single input, it's always length 1 and clutters indexing. Pass `remove_batch_dim=True` to strip it.

In [13]:
# Classic GPT-2 paper abstract opening sentence
gpt2_text = (
    "Natural language processing tasks, such as question answering, machine translation, "
    "reading comprehension, and summarization, are typically approached with supervised "
    "learning on taskspecific datasets."
)    

# Tokenize the text → integer token IDs
gpt2_tokens = model.to_tokens(gpt2_text)
print("Token tensor device:", gpt2_tokens.device)
print("Token tensor shape: [batch, seq_len] =", gpt2_tokens.shape)

# Run with cache: returns (logits, ActivationCache)
gpt2_logits, gpt2_cache = model.run_with_cache(gpt2_tokens, remove_batch_dim=True)

Token tensor device: cuda:0
Token tensor shape: [batch, seq_len] = torch.Size([1, 33])


In [14]:
# Inspect the attention pattern of all heads in layer 0
# Shape: [n_heads, dest_position, src_position]
print("Cache type:", type(gpt2_cache))
attention_pattern = gpt2_cache["pattern", 0, "attn"]
print("Attention pattern shape:", attention_pattern.shape)

# Convert token IDs back to human-readable strings
gpt2_str_tokens = model.to_str_tokens(gpt2_text)
print("Tokens:", gpt2_str_tokens)

Cache type: <class 'transformer_lens.ActivationCache.ActivationCache'>
Attention pattern shape: torch.Size([12, 33, 33])
Tokens: ['<|endoftext|>', 'Natural', ' language', ' processing', ' tasks', ',', ' such', ' as', ' question', ' answering', ',', ' machine', ' translation', ',', ' reading', ' comprehension', ',', ' and', ' summar', 'ization', ',', ' are', ' typically', ' approached', ' with', ' supervised', ' learning', ' on', ' tasks', 'pe', 'cific', ' datasets', '.']


In [15]:
# Visualize attention heads for layer 0 using CircuitsVis.
# The grid is lower triangular because GPT-2 uses CAUSAL attention
# (each token can only attend to earlier tokens).
# Try hovering over the heads to see which positions they focus on!
print("Layer 0 Head Attention Patterns:")
cv.attention.attention_heads(
    tokens=gpt2_str_tokens,
    attention=attention_pattern
)

Layer 0 Head Attention Patterns:


In [16]:
# MEMORY-EFFICIENT ALTERNATIVE:
# If you only need layer 0 attention, don't cache everything.
# stop_at_layer=1  → stops the forward pass after layer 0 (saves compute)
# names_filter     → only caches the activation you ask for (saves memory)

attn_hook_name = "blocks.0.attn.hook_pattern"
attn_layer = 0

_, gpt2_attn_cache = model.run_with_cache(
    gpt2_tokens,
    remove_batch_dim=True,
    stop_at_layer=attn_layer + 1,
    names_filter=[attn_hook_name]
)

gpt2_attn = gpt2_attn_cache[attn_hook_name]
assert torch.allclose(gpt2_attn, attention_pattern), "Values should match the full cache!"
print("✓ Filtered cache gives the same result as the full cache.")

✓ Filtered cache gives the same result as the full cache.


---
## 4. Hooks: Intervening on Activations

This is the core power of TransformerLens. **Hook points** are named locations wrapping every internal activation. You can:
- **Read** an activation value mid-forward-pass
- **Edit** (ablate, patch, zero out) an activation mid-forward-pass

A **hook function** has this signature:
```python
def my_hook(activation_tensor, hook: HookPoint):
    # Do something to activation_tensor
    return modified_tensor
```

You attach it temporarily for a single forward pass using `model.run_with_hooks()`.

**Why is this better than PyTorch hooks?**
- Hooks act on *activations*, not layers → finer granularity
- Named positions make it obvious exactly *where* the intervention happens
- `model.reset_hooks()` cleanly removes all hooks
- Hooks are automatically cleaned up after `run_with_hooks()`

---
### 4a. Head Ablation

**Ablation** = zeroing out a specific component to see what effect it has on model output. The go-to causal intervention in mechanistic interpretability.

In [17]:
layer_to_ablate = 0
head_index_to_ablate = 8

# Hook function: zero out all value vectors for head 8 in layer 0
# Type annotations are documentation-only, not enforced
def head_ablation_hook(
    value: Float[torch.Tensor, "batch pos head_index d_head"],
    hook: HookPoint
) -> Float[torch.Tensor, "batch pos head_index d_head"]:
    print(f"  Value tensor shape: {value.shape}")
    value[:, :, head_index_to_ablate, :] = 0.0
    return value

# Run WITHOUT the hook (baseline)
original_loss = model(gpt2_tokens, return_type="loss")

# Run WITH the ablation hook
# utils.get_act_name("v", layer) → the canonical name for the value activation at that layer
ablated_loss = model.run_with_hooks(
    gpt2_tokens,
    return_type="loss",
    fwd_hooks=[(utils.get_act_name("v", layer_to_ablate), head_ablation_hook)]
)

print(f"\nOriginal Loss:  {original_loss.item():.3f}")
print(f"Ablated Loss:   {ablated_loss.item():.3f}")
print(f"Loss increase:  {(ablated_loss - original_loss).item():.3f}")
print("\n(A large increase means that head 8 in layer 0 was contributing meaningfully to predictions)")

  Value tensor shape: torch.Size([1, 33, 12, 64])

Original Loss:  3.999
Ablated Loss:   5.453
Loss increase:  1.454

(A large increase means that head 8 in layer 0 was contributing meaningfully to predictions)


---
### 4b. Activation Patching on the Indirect Object Identification (IOI) Task

**What is the IOI task?**  
Sentences like *"After John and Mary went to the store, **Mary** gave a bottle of milk to"* should be completed with *" John"* (the indirect object). GPT-2 can do this — we want to understand *which internal activations are responsible*.

**Activation Patching** (from ROME paper, Meng & Bau):  
1. Define a **clean prompt** (original, correct) and a **corrupted prompt** (swapped name, flips the answer)
2. Run the model on the corrupted prompt
3. For each activation (layer × position), *patch in* the clean value at that location
4. Measure how much performance recovers → tells us how important that activation is

We patch the **residual stream** (`resid_pre`) at every layer/position combination and record the logit difference (P(John) - P(Mary)) for each patch.

In [18]:
# The clean prompt → answer should be " John"
# The corrupted prompt → same structure, but the giving person is now "John",
# making the expected answer ambiguous/flipped
clean_prompt = "After John and Mary went to the store, Mary gave a bottle of milk to"
corrupted_prompt = "After John and Mary went to the store, John gave a bottle of milk to"

clean_tokens = model.to_tokens(clean_prompt)
corrupted_tokens = model.to_tokens(corrupted_prompt)

def logits_to_logit_diff(logits, correct_answer=" John", incorrect_answer=" Mary"):
    """Compute P(correct) - P(incorrect) on the final token position."""
    # to_single_token maps a string to a single token id (raises if multi-token)
    correct_index = model.to_single_token(correct_answer)
    incorrect_index = model.to_single_token(incorrect_answer)
    return logits[0, -1, correct_index] - logits[0, -1, incorrect_index]

# Run on clean prompt with cache (we'll patch these activations in later)
clean_logits, clean_cache = model.run_with_cache(clean_tokens)
clean_logit_diff = logits_to_logit_diff(clean_logits)
print(f"Clean logit difference:     {clean_logit_diff.item():.3f}  (positive → model predicts John)")

# Run on corrupted prompt (no cache needed)
corrupted_logits = model(corrupted_tokens)
corrupted_logit_diff = logits_to_logit_diff(corrupted_logits)
print(f"Corrupted logit difference: {corrupted_logit_diff.item():.3f}  (negative → model predicts Mary)")

Clean logit difference:     4.276  (positive → model predicts John)
Corrupted logit difference: -2.738  (negative → model predicts Mary)


In [19]:
# Hook function for activation patching.
# We patch in the clean residual stream at a specific position.
# We use functools.partial to fix the 'position' argument before passing to run_with_hooks.

def residual_stream_patching_hook(
    resid_pre: Float[torch.Tensor, "batch pos d_model"],
    hook: HookPoint,
    position: int
) -> Float[torch.Tensor, "batch pos d_model"]:
    # hook.name gives the full activation name (e.g. "blocks.7.hook_resid_pre")
    clean_resid_pre = clean_cache[hook.name]
    resid_pre[:, position, :] = clean_resid_pre[:, position, :]
    return resid_pre

# Store results: shape [n_layers, n_positions]
num_positions = len(clean_tokens[0])
ioi_patching_result = torch.zeros(
    (model.cfg.n_layers, num_positions),
    device=model.cfg.device
)

# Iterate over every (layer, position) pair
# This takes ~1-2 minutes on CPU, ~10-20s on GPU
for layer in tqdm.tqdm(range(model.cfg.n_layers), desc="Patching layers"):
    for position in range(num_positions):
        temp_hook_fn = partial(residual_stream_patching_hook, position=position)
        patched_logits = model.run_with_hooks(
            corrupted_tokens,
            fwd_hooks=[(utils.get_act_name("in", layer), temp_hook_fn)]
        )
        patched_logit_diff = logits_to_logit_diff(patched_logits).detach()
        # Normalize to [0, 1]: 0 = no recovery, 1 = full recovery to clean performance
        ioi_patching_result[layer, position] = (
            (patched_logit_diff - corrupted_logit_diff)
            / (clean_logit_diff - corrupted_logit_diff)
        )

print("Patching complete!")

Patching layers: 100%|██████████| 12/12 [00:04<00:00,  2.70it/s]

Patching complete!


In [20]:
# Visualize the patching results as a heatmap.
# Red = patching here mostly recovers clean performance (important location)
# Blue = patching here hurts performance (the clean activation at this location was not helpful)
#
# Key observations:
# - Early layers: only the second subject token (Mary/John) position matters
# - Layers 7-8: information moves from the subject token to the FINAL token
#   (this is where the S-Inhibition and Name Mover heads operate)

token_labels = [
    f"{token}_{index}"
    for index, token in enumerate(model.to_str_tokens(clean_tokens))
]

imshow(
    ioi_patching_result,
    x=token_labels,
    xaxis="Position",
    yaxis="Layer",
    title="Normalized Logit Difference After Patching Residual Stream on the IOI Task",
)

---
## 5. Hooks: Accessing Activations — Induction Heads

Hooks don't have to *change* activations — they can just *read* them. If the hook returns `None`, the activation is left unchanged.

**Induction Heads** are one of the most important and well-studied circuits in language models. They:
1. Detect when a token appeared earlier in context
2. Predict that the *next* token will be whatever followed that earlier occurrence

They're implemented by **two heads in different layers that compose**:
- **Previous token head** (earlier layer): attends to the previous token
- **Induction head** (later layer): attends to the token *after* a previous copy of the current token

**Test:** Generate 50 random tokens, then repeat them. Loss should drop sharply at position 50 — the model can now use induction to predict the second half exactly.

**Induction score:** For each attention head, average how much attention it pays to the token 49 positions back (the position that an induction head *should* focus on).

In [21]:
# Generate random token sequences and repeat them: [A B C ... A B C ...]
batch_size = 10
seq_len = 50

input_tensor = torch.randint(1000, 10000, (batch_size, seq_len))
random_tokens = input_tensor.to(model.cfg.device)

# Repeat each sequence: [A B ... Z A B ... Z]
repeated_tokens = einops.repeat(random_tokens, "batch seq_len -> batch (2 seq_len)")

# Compute loss at each position
repeated_logits = model(repeated_tokens)
correct_log_probs = model.loss_fn(repeated_logits, repeated_tokens, per_token=True)
loss_by_position = einops.reduce(correct_log_probs, "batch position -> position", "mean")

line(
    loss_by_position,
    xaxis="Position",
    yaxis="Loss",
    title="Loss by Position on Random Repeated Tokens (drop at pos 50 = induction!)",
)

In [22]:
# Compute the "induction score" for every head in every layer.
# Score = average attention paid to the position (seq_len - 1) tokens back.
# Induction heads should score close to 1.0; other heads near 0.

induction_score_store = torch.zeros(
    (model.cfg.n_layers, model.cfg.n_heads),
    device=model.cfg.device
)

def induction_score_hook(
    pattern: Float[torch.Tensor, "batch head_index dest_pos source_pos"],
    hook: HookPoint,
):
    # Diagonal at offset (1 - seq_len) captures attention from position i to position i - (seq_len - 1)
    # This is the "induction stripe" — the diagonal an induction head attends along
    induction_stripe = pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len)
    induction_score = einops.reduce(
        induction_stripe, "batch head_index position -> head_index", "mean"
    )
    # hook.layer() extracts the layer number from the hook name string
    induction_score_store[hook.layer(), :] = induction_score

# Add hook to every layer's attention pattern simultaneously using a name filter
pattern_hook_names_filter = lambda name: name.endswith("pattern")

model.run_with_hooks(
    repeated_tokens,
    return_type=None,   # We only want side effects (writing to induction_score_store)
    fwd_hooks=[(pattern_hook_names_filter, induction_score_hook)]
)

imshow(
    induction_score_store,
    xaxis="Head",
    yaxis="Layer",
    title="Induction Score by Head (bright red = strong induction head)"
)

In [23]:
# The heatmap shows Head 5 in Layer 5 has a very high induction score.
# Let's directly visualize its attention pattern on a short repeated sequence
# to confirm the induction stripe.

if IN_GITHUB:
    torch.manual_seed(50)

induction_head_layer = 5
induction_head_index = 5

single_random_sequence = torch.randint(1000, 10000, (1, 20)).to(model.cfg.device)
repeated_random_sequence = einops.repeat(
    single_random_sequence, "batch seq_len -> batch (2 seq_len)"
)

def visualize_pattern_hook(
    pattern: Float[torch.Tensor, "batch head_index dest_pos source_pos"],
    hook: HookPoint,
):
    display(
        cv.attention.attention_pattern(
            tokens=model.to_str_tokens(repeated_random_sequence),
            attention=pattern[0, induction_head_index, :, :],
        )
    )

model.run_with_hooks(
    repeated_random_sequence,
    return_type=None,
    fwd_hooks=[(utils.get_act_name("pattern", induction_head_layer), visualize_pattern_hook)]
)
# You should see the classic induction stripe: a diagonal band offset 19 tokens back

---
## 6. Available Models

TransformerLens supports 9,000+ models. You swap models by just changing the string in `boot_transformers`. Let's replicate the induction head analysis on **DistilGPT-2** (half the layers of GPT-2).

### Key model families available:

| Model | Size | Notes |
|-------|------|-------|
| GPT-2 Small | 85M | Best for learning — small, well-studied |
| GPT-2 Medium | 300M | |
| GPT-2 Large | 700M | |
| GPT-2 XL | 1.5B | |
| GPT-Neo 125M / 1.3B / 2.7B | — | Trained on the Pile |
| GPT-J | 6B | Eleuther |
| GPT-NeoX | 20B | Eleuther |
| OPT 125M→66B | — | Meta |
| StableLM 3B / 7B | — | Stability AI |
| BERT Base | 108M | Encoder-only |
| **Toy models** (`gelu-1l` to `gelu-4l`) | 1-4 layers | Easiest to interpret! |
| SoLU models (`solu-1l` to `solu-12l`) | 13M–340M | MLP neurons more interpretable |

Toy models are especially good for research practice — they're fast to run and their circuits are more tractable to reverse-engineer.

In [24]:
# Load DistilGPT-2 — same API, different model
distilgpt2 = TransformerBridge.boot_transformers("distilgpt2", device=device)
print(f"DistilGPT-2 config: {distilgpt2.cfg.n_layers} layers, {distilgpt2.cfg.n_heads} heads")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 3600.87it/s]


DistilGPT-2 config: 6 layers, 12 heads


In [25]:
# Exact same induction head analysis — just swap the model object
distilgpt2_induction_score_store = torch.zeros(
    (distilgpt2.cfg.n_layers, distilgpt2.cfg.n_heads),
    device=distilgpt2.cfg.device
)

def induction_score_hook_distil(
    pattern: Float[torch.Tensor, "batch head_index dest_pos source_pos"],
    hook: HookPoint,
):
    induction_stripe = pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len)
    induction_score = einops.reduce(
        induction_stripe, "batch head_index position -> head_index", "mean"
    )
    distilgpt2_induction_score_store[hook.layer(), :] = induction_score

pattern_hook_names_filter = lambda name: name.endswith("pattern")

distilgpt2.run_with_hooks(
    repeated_tokens,
    return_type=None,
    fwd_hooks=[(pattern_hook_names_filter, induction_score_hook_distil)]
)

imshow(
    distilgpt2_induction_score_store,
    xaxis="Head",
    yaxis="Layer",
    title="Induction Score by Head in DistilGPT-2"
)

---
## 7. Model Architecture Conventions & Parameter Names

TransformerLens presents GPT-2 in a cleaned-up layout that's easier to analyse than HuggingFace's internal format. The key changes:

### Attention weight matrices
- **HuggingFace:** W_Q, W_K, W_V are one big concatenated matrix `[d_model, 3*d_model]`
- **TransformerLens:** They are **3 separate matrices**, each `[n_heads, d_model, d_head]`

### Activation shapes
All attention activations have a **separate head axis**:
- Keys, queries, values, z: `[batch, position, head_index, d_head]`
- W_O (output projection): `[head_index, d_head, d_model]`

### Naming convention for `utils.get_act_name()`

| Name | What it refers to |
|------|-----------------|
| `"q"` | Query vectors |
| `"k"` | Key vectors |
| `"v"` | Value vectors |
| `"z"` | Output of each head before W_O |
| `"pattern"` | Attention weights (post softmax) |
| `"attn_out"` | Output of the full attention block |
| `"mlp_out"` | Output of the MLP block |
| `"resid_pre"` | Residual stream before layer |
| `"resid_post"` | Residual stream after layer |
| `"resid_mid"` | Residual stream between attn and MLP |

### Key hyperparameters (GPT-2 Small)

| Param | Value | Meaning |
|-------|-------|--------|
| `n_layers` | 12 | Number of transformer blocks |
| `n_heads` | 12 | Attention heads per layer |
| `d_model` | 768 | Residual stream width |
| `d_head` | 64 | d_model / n_heads |
| `d_mlp` | 3072 | MLP hidden dimension (4 × d_model) |
| `d_vocab` | 50257 | Vocabulary size |
| `n_ctx` | 1024 | Maximum context length |

In [26]:
# Inspect the model config programmatically
cfg = model.cfg
print(f"Model: GPT-2 Small")
print(f"  n_layers : {cfg.n_layers}")
print(f"  n_heads  : {cfg.n_heads}")
print(f"  d_model  : {cfg.d_model}")
print(f"  d_head   : {cfg.d_head}")
print(f"  d_mlp    : {cfg.d_mlp}")
print(f"  d_vocab  : {cfg.d_vocab}")
print(f"  n_ctx    : {cfg.n_ctx}")
print(f"  device   : {cfg.device}")

Model: GPT-2 Small
  n_layers : 12
  n_heads  : 12
  d_model  : 768
  d_head   : 64
  d_mlp    : 3072
  d_vocab  : 50257
  n_ctx    : 1024
  device   : cuda


In [27]:
# List all named parameters and their shapes
# Useful for knowing what weights you can access directly
print("First 30 named parameters and shapes:")
print("-" * 60)
for i, (name, param) in enumerate(model.named_parameters()):
    print(f"{name:45s} {list(param.shape)}")
    if i >= 29:
        print(f"... (and more)")
        break

First 30 named parameters and shapes:
------------------------------------------------------------
transformer.wte._original_component.weight    [50257, 768]
transformer.wpe._original_component.weight    [1024, 768]
transformer.h.0._original_component.ln_1._original_component.weight [768]
transformer.h.0._original_component.ln_1._original_component.bias [768]
transformer.h.0._original_component.attn.q._original_component.weight [768, 768]
transformer.h.0._original_component.attn.q._original_component.bias [768]
transformer.h.0._original_component.attn.k._original_component.weight [768, 768]
transformer.h.0._original_component.attn.k._original_component.bias [768]
transformer.h.0._original_component.attn.v._original_component.weight [768, 768]
transformer.h.0._original_component.attn.v._original_component.bias [768]
transformer.h.0._original_component.attn.qkv._original_component.weight [768, 2304]
transformer.h.0._original_component.attn.qkv._original_component.bias [2304]
transformer.

In [28]:
# You can access individual weight matrices directly
# Shape: [n_heads, d_model, d_head]
W_Q_layer0 = model.W_Q[0]  # Query weight for layer 0
W_K_layer0 = model.W_K[0]
W_V_layer0 = model.W_V[0]
W_O_layer0 = model.W_O[0]  # Shape: [n_heads, d_head, d_model]

print(f"W_Q layer 0: {W_Q_layer0.shape}  (n_heads={cfg.n_heads}, d_model={cfg.d_model}, d_head={cfg.d_head})")
print(f"W_K layer 0: {W_K_layer0.shape}")
print(f"W_V layer 0: {W_V_layer0.shape}")
print(f"W_O layer 0: {W_O_layer0.shape}  (n_heads, d_head, d_model)")

print()
print("MLP weights:")
print(f"W_in  layer 0: {model.W_in[0].shape}   (d_model → d_mlp)")
print(f"W_out layer 0: {model.W_out[0].shape}  (d_mlp → d_model)")

W_Q layer 0: torch.Size([12, 768, 64])  (n_heads=12, d_model=768, d_head=64)
W_K layer 0: torch.Size([12, 768, 64])
W_V layer 0: torch.Size([12, 768, 64])
W_O layer 0: torch.Size([12, 64, 768])  (n_heads, d_head, d_model)

MLP weights:
W_in  layer 0: torch.Size([768, 3072])   (d_model → d_mlp)
W_out layer 0: torch.Size([3072, 768])  (d_mlp → d_model)


---
## Further Reading & Next Steps

Now that you know the core TransformerLens primitives — `run_with_cache`, `run_with_hooks`, activation patching, and induction heads — here's what to explore next:

### Official Resources
- 📖 [ARENA Chapter 1: Transformer Interpretability](https://learn.arena.education/chapter1_transformer_interp/) — structured curriculum with exercises
- 📝 [200 Concrete Open Problems in Mech Interp](https://neelnanda.io/concrete-open-problems) — research directions to pursue
- 🔍 [Exploratory Analysis Demo](https://neelnanda.io/exploratory-analysis-demo) — Neel's live research process
- 📹 [Recorded research walkthrough](https://www.youtube.com/watch?v=yo4QvDn-vsU)

### Key Papers to Read
- [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html) (Elhage et al.)
- [In-context Learning and Induction Heads](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) (Olsson et al.)
- [Interpretability in the Wild: IOI](https://arxiv.org/abs/2211.00593) (Wang et al., Redwood Research)
- [ROME: Locating and Editing Factual Associations](https://rome.baulab.info/) (Meng & Bau)

### What to Build Next
1. **Direct Logit Attribution**: Decompose each head's contribution to the final logits
2. **Neuron analysis**: Which neurons activate for what inputs?
3. **Copy suppression heads**: Why do some heads actively suppress repeating the context?
4. **QK and OV circuits**: Use the `FactoredMatrix` class to analyse attention head circuits directly from weights

```python
# Quick teaser — the OV circuit of a head:
# W_OV = W_V @ W_O = how does attending to a token affect the output?
W_OV = FactoredMatrix(model.W_V[0, 5], model.W_O[0, 5])  # layer 0, head 5
```

In [29]:
# Peak VRAM used by this kernel so far (PyTorch allocator only; nvidia-smi will show a bit more
# because of the CUDA context). Restart the kernel to release GPU memory before opening another notebook.
if torch.cuda.is_available():
    print(f"Peak allocated : {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
    print(f"Currently held : {torch.cuda.memory_reserved() / 2**30:.2f} GiB (reserved by PyTorch cache)")

Peak allocated : 1.97 GiB
Currently held : 2.52 GiB (reserved by PyTorch cache)
